# LeetCode 1046 — Last Stone Weight bằng Python

**Link đề:** https://leetcode.com/problems/last-stone-weight/
**Độ khó:** Easy · **Chủ đề:** Array, Heap (Priority Queue — hàng đợi ưu tiên), Simulation

```python
class Solution:
    def lastStoneWeight(self, stones: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Đọc đúng đề: mỗi lượt đập **hai viên nặng nhất**, viên còn lại (nếu có) nặng `y - x`.
2. Hiểu bài toán **mô phỏng** "lặp lại: lấy phần tử lớn nhất" và vì sao cấu trúc dữ liệu quyết định tốc độ.
3. Học **heap** từ đầu: `heapq` là min-heap, mẹo **đổi dấu** để có max-heap; `heapify`, `heappush`, `heappop`, `heapreplace`.
4. Viết lời giải từ dễ đến tinh: sort lại mỗi lượt → chèn có thứ tự bằng `bisect` → max-heap → bucket theo cân nặng.
5. Trace heap từng bước, chứng minh mô phỏng kết thúc, phân tích Time/Space, so sánh 4 cách.
6. Làm bài tập biến thể (Last Stone Weight II là bài khác hẳn — DP; K viên nặng nhất; lấy quà từ đống giàu nhất).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho mảng `stones`, `stones[i]` là cân nặng viên đá thứ `i`. Mỗi lượt: chọn **hai viên nặng nhất** `x <= y` và đập vào nhau:
- `x == y`: cả hai bị phá hủy.
- `x != y`: viên `x` bị phá hủy, viên `y` còn lại với cân nặng `y - x`.

Cuối cùng còn **nhiều nhất một** viên. Trả về cân nặng viên đó, hoặc `0` nếu không còn viên nào.

**Ví dụ 1:** `stones = [2,7,4,1,8,1]` → `1`

| Lượt | Hai viên nặng nhất | Kết quả | Mảng còn lại |
|:-:|:-:|---|---|
| 1 | 7, 8 | còn 8 − 7 = 1 | `[2,4,1,1,1]` |
| 2 | 2, 4 | còn 4 − 2 = 2 | `[2,1,1,1]` |
| 3 | 1, 2 | còn 2 − 1 = 1 | `[1,1,1]` |
| 4 | 1, 1 | phá hủy cả hai | `[1]` |

**Ví dụ 2:** `stones = [1]` → `1` (không có lượt nào).

**Constraints:** `1 <= stones.length <= 30`, `1 <= stones[i] <= 1000`.

Gợi ý của đề: *mô phỏng quá trình, bằng heap hoặc sắp xếp lại mỗi lượt.*

In [1]:
# Mô phỏng ví dụ 1 bằng tay, dùng max() và remove()
stones = [2, 7, 4, 1, 8, 1]
turn = 1
while len(stones) > 1:
    y = max(stones); stones.remove(y)
    x = max(stones); stones.remove(x)
    if y != x:
        stones.append(y - x)
    print(f"lượt {turn}: đập {x} và {y} -> {stones}")
    turn += 1
print("Kết quả:", stones[0] if stones else 0)

lượt 1: đập 7 và 8 -> [2, 4, 1, 1, 1]
lượt 2: đập 2 và 4 -> [1, 1, 1, 2]
lượt 3: đập 1 và 2 -> [1, 1, 1]
lượt 4: đập 1 và 1 -> [1]
Kết quả: 1


- `stones.remove(v)` xóa **lần xuất hiện đầu tiên** của giá trị `v` (O(n)).
- `max(stones)` duyệt cả list (O(n)).
- `stones[0] if stones else 0`: nếu list khác rỗng lấy phần tử đầu, ngược lại 0.

## 2. Nền tảng: heap (hàng đợi ưu tiên)

### 2.1. Heap là gì?

**Min-heap** là một cây nhị phân "gần đầy" lưu trong list sao cho **mỗi node ≤ các con của nó** → phần tử nhỏ nhất luôn ở **gốc** `h[0]`.

Trong list: node ở chỉ số `i` có con ở `2i + 1` và `2i + 2`, cha ở `(i - 1) // 2`.

| Thao tác | Ý nghĩa | Độ phức tạp |
|---|---|---|
| `heapq.heapify(h)` | biến list thành heap tại chỗ | O(n) |
| `heapq.heappush(h, x)` | thêm `x` | O(log n) |
| `heapq.heappop(h)` | lấy ra phần tử nhỏ nhất | O(log n) |
| `h[0]` | xem phần tử nhỏ nhất | O(1) |
| `heapq.heapreplace(h, x)` | pop nhỏ nhất **rồi** push `x` | O(log n) |
| `heapq.heappushpop(h, x)` | push `x` **rồi** pop nhỏ nhất | O(log n) |

In [2]:
import heapq

h = [5, 3, 8, 1, 9, 2]
heapq.heapify(h)
print("sau heapify:", h, "| nhỏ nhất h[0] =", h[0])
for i in range(len(h)):
    kids = [h[j] for j in (2 * i + 1, 2 * i + 2) if j < len(h)]
    print(f"  node {i} = {h[i]}, con = {kids}")

sau heapify: [1, 3, 2, 5, 9, 8] | nhỏ nhất h[0] = 1
  node 0 = 1, con = [3, 2]
  node 1 = 3, con = [5, 9]
  node 2 = 2, con = [8]
  node 3 = 5, con = []
  node 4 = 9, con = []
  node 5 = 8, con = []


Chú ý: heap **không** phải list đã sắp xếp — chỉ đảm bảo cha ≤ con. Nhưng `heappop` liên tục sẽ cho các phần tử theo thứ tự tăng dần.

In [3]:
h2 = h[:]
out = []
while h2:
    out.append(heapq.heappop(h2))
print("heappop liên tục:", out)

heappop liên tục: [1, 2, 3, 5, 8, 9]


### 2.2. Max-heap bằng mẹo **đổi dấu**

`heapq` chỉ có min-heap. Muốn lấy **lớn nhất**, lưu `-x` thay cho `x`: số lớn nhất thành số âm nhỏ nhất → nằm ở gốc. Khi lấy ra nhớ đổi dấu lại.

In [4]:
stones = [2, 7, 4, 1, 8, 1]
max_heap = [-s for s in stones]
heapq.heapify(max_heap)
print("heap lưu số âm:", max_heap)
print("lớn nhất =", -max_heap[0])
print("pop lần lượt:", [-heapq.heappop(max_heap) for _ in range(3)])

heap lưu số âm: [-8, -7, -4, -1, -2, -1]
lớn nhất = 8
pop lần lượt: [8, 7, 4]


### 2.3. `bisect.insort` — chèn giữ list luôn sắp xếp

`bisect.insort(a, x)` chèn `x` vào list `a` **đã sắp xếp** sao cho vẫn sắp xếp. Tìm vị trí bằng tìm kiếm nhị phân O(log n), nhưng **dịch phần tử** để chèn tốn O(n).

In [5]:
import bisect
a = [1, 2, 4, 7]
bisect.insort(a, 5)
print(a)
print("pop() lấy phần tử cuối (lớn nhất):", a.pop(), "| còn:", a)

[1, 2, 4, 5, 7]
pop() lấy phần tử cuối (lớn nhất): 7 | còn: [1, 2, 4, 5]


## 3. Cách trực giác / Baseline

### 3.1. Cách 1 — Sắp xếp lại mỗi lượt

Mỗi lượt: `sort()` rồi lấy hai phần tử cuối (lớn nhất).

In [6]:
def last_stone_sort(stones: list[int]) -> int:
    stones = stones[:]                   # bản sao, không phá input
    while len(stones) > 1:
        stones.sort()
        y = stones.pop()                 # lớn nhất
        x = stones.pop()                 # lớn nhì
        if y != x:
            stones.append(y - x)
    return stones[0] if stones else 0

print(last_stone_sort([2, 7, 4, 1, 8, 1]))   # 1
print(last_stone_sort([1]))                   # 1
print(last_stone_sort([3, 3]))                # 0

1
1
0


Mỗi lượt số viên giảm ít nhất 1 → tối đa `n - 1` lượt; mỗi lượt sort O(n log n) → **O(n² log n)**. Với `n ≤ 30` thì vẫn tức thì. Nhưng nếu `n = 10^5` thì không chịu nổi.

### 3.2. Cách 2 — Giữ list luôn sắp xếp, chèn bằng `bisect.insort`

Sort một lần ở đầu; sau mỗi lần đập, chèn mảnh còn lại vào **đúng chỗ** thay vì sort lại.

In [7]:
def last_stone_insort(stones: list[int]) -> int:
    arr = sorted(stones)
    while len(arr) > 1:
        y = arr.pop()
        x = arr.pop()
        if y != x:
            bisect.insort(arr, y - x)
    return arr[0] if arr else 0

print(last_stone_insort([2, 7, 4, 1, 8, 1]), last_stone_insort([1]), last_stone_insort([3, 3]))

1 1 0


Mỗi lượt: `pop()` cuối O(1), `insort` O(n) (do dịch phần tử) → tổng **O(n²)**.

## 4. Nút thắt & insight

Thao tác lặp lại là: **lấy 2 phần tử lớn nhất** và **thêm 1 phần tử mới**. Ta không cần cả list sắp xếp — chỉ cần lấy nhanh **max**.

Đó chính là việc của **heap**: lấy max O(log n), thêm O(log n). Tổng **O(n log n)**.

| Cấu trúc | Lấy max | Thêm | Tổng cho ~n lượt |
|---|---|---|---|
| list chưa sắp xếp | O(n) | O(1) | O(n²) |
| list sắp xếp (sort lại) | O(1) | O(n log n) | O(n² log n) |
| list sắp xếp (insort) | O(1) | O(n) | O(n²) |
| **max-heap** | **O(log n)** | **O(log n)** | **O(n log n)** |

## 5. Lời giải chính — max-heap bằng `heapq` và số âm

```python
heap = [-s for s in stones]          # Đoạn 1: đổi dấu
heapq.heapify(heap)                  #          biến thành heap O(n)
while len(heap) > 1:                 # Đoạn 2: còn ít nhất 2 viên
    y = -heapq.heappop(heap)         #   nặng nhất
    x = -heapq.heappop(heap)         #   nặng nhì  (x <= y)
    if y != x:
        heapq.heappush(heap, -(y - x))   # mảnh còn lại
return -heap[0] if heap else 0       # Đoạn 3: còn 1 viên hoặc 0 viên
```

Chú ý dấu: lưu `-(y - x)` = `x - y` (số âm), vì heap lưu cân nặng **âm**.

In [8]:
def last_stone_heap(stones: list[int]) -> int:
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        y = -heapq.heappop(heap)
        x = -heapq.heappop(heap)
        if y != x:
            heapq.heappush(heap, -(y - x))
    return -heap[0] if heap else 0

print(last_stone_heap([2, 7, 4, 1, 8, 1]), last_stone_heap([1]), last_stone_heap([3, 3]))

1 1 0


### 5.1. Biến tấu: dùng `heapreplace` để bớt một thao tác

Khi `y != x`: thay vì pop `x` rồi push `y - x`, ta có thể **xem** `x` ở `heap[0]` và `heapreplace` nó bằng mảnh mới (pop + push trong một lần sift).

In [9]:
def last_stone_heapreplace(stones: list[int]) -> int:
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        y = -heapq.heappop(heap)
        x = -heap[0]                     # nặng nhì, chưa lấy ra
        if y == x:
            heapq.heappop(heap)
        else:
            heapq.heapreplace(heap, -(y - x))
    return -heap[0] if heap else 0

print(last_stone_heapreplace([2, 7, 4, 1, 8, 1]), last_stone_heapreplace([1]), last_stone_heapreplace([3, 3]))

1 1 0


### 5.2. Cách 4 — Bucket theo cân nặng (vì cân nặng ≤ 1000)

`count[w]` = số viên nặng `w`. Duyệt `w` từ lớn xuống nhỏ:
- Nếu `count[w] ≥ 2`: các cặp viên cùng nặng `w` đập nhau và **biến mất hết** → chỉ giữ `count[w] % 2`.
- Nếu còn đúng 1 viên `w`: tìm viên nặng nhì `w2 < w` (dò xuống); đập → thêm 1 viên nặng `w - w2` (luôn `< w`, nên sẽ được xử lý sau khi duyệt xuống).

Đây là cách cho thấy: khi **miền giá trị nhỏ**, đếm có thể thay heap.

In [10]:
def last_stone_bucket(stones: list[int]) -> int:
    W = max(stones)
    count = [0] * (W + 1)
    for s in stones:
        count[s] += 1
    w = W
    while w > 0:
        count[w] %= 2                    # các cặp bằng nhau tự triệt tiêu
        if count[w] == 0:
            w -= 1
            continue
        # còn đúng một viên nặng w: tìm viên nặng nhì
        w2 = w - 1
        while w2 > 0 and count[w2] == 0:
            w2 -= 1
        if w2 == 0:                      # không còn viên nào khác
            return w
        count[w] = 0
        count[w2] -= 1
        count[w - w2] += 1               # mảnh còn lại, nhẹ hơn w
        w = max(w2, w - w2)              # viên nặng nhất hiện tại
    return 0

print(last_stone_bucket([2, 7, 4, 1, 8, 1]), last_stone_bucket([1]), last_stone_bucket([3, 3]))

1 1 0


Vì sao `w = max(w2, w - w2)`? Sau khi đập, viên nặng nhất còn lại hoặc là các viên nặng `w2` (nếu còn), hoặc là mảnh mới `w - w2`. Mọi viên khác đều `< w2`. Vòng `while` sẽ tự dò xuống nếu thùng `w` hiện tại rỗng.

Độ phức tạp: con trỏ `w`, `w2` chủ yếu đi xuống; mỗi lượt đập có thể đưa `w` về tối đa `w - w2`… Tổng quát `O(n·W)` tệ nhất, nhưng thực tế nhanh vì `W ≤ 1000`, `n ≤ 30`. Ta giữ cách này như ví dụ về tư duy "đếm" — **cách heap vẫn là lời giải chuẩn**.

## 6. TRACE heap từng bước

Ví dụ 1 `[2,7,4,1,8,1]`. Cột "heap (đổi lại dấu, sắp xếp để dễ đọc)" cho thấy các viên còn lại.

| Lượt | pop y | pop x | y ≠ x? | push | Các viên còn lại |
|:-:|:-:|:-:|:-:|:-:|---|
| bắt đầu | | | | | `8 7 4 2 1 1` |
| 1 | 8 | 7 | ✓ | 1 | `4 2 1 1 1` |
| 2 | 4 | 2 | ✓ | 2 | `2 1 1 1` |
| 3 | 2 | 1 | ✓ | 1 | `1 1 1` |
| 4 | 1 | 1 | ✗ | — | `1` |

Còn 1 viên → trả `1`.

In [11]:
def trace_heap(stones):
    heap = [-s for s in stones]
    heapq.heapify(heap)
    show = lambda: sorted((-v for v in heap), reverse=True)
    print("bắt đầu: heap nội bộ =", heap, "| các viên =", show())
    turn = 1
    while len(heap) > 1:
        y = -heapq.heappop(heap)
        x = -heapq.heappop(heap)
        if y != x:
            heapq.heappush(heap, -(y - x))
            action = f"push {y - x}"
        else:
            action = "cả hai vỡ"
        print(f"lượt {turn}: y={y}, x={x} -> {action:<10} | heap nội bộ = {heap} | các viên = {show()}")
        turn += 1
    result = -heap[0] if heap else 0
    print("Kết quả:", result)
    return result

trace_heap([2, 7, 4, 1, 8, 1])

bắt đầu: heap nội bộ = [-8, -7, -4, -1, -2, -1] | các viên = [8, 7, 4, 2, 1, 1]
lượt 1: y=8, x=7 -> push 1     | heap nội bộ = [-4, -2, -1, -1, -1] | các viên = [4, 2, 1, 1, 1]
lượt 2: y=4, x=2 -> push 2     | heap nội bộ = [-2, -1, -1, -1] | các viên = [2, 1, 1, 1]
lượt 3: y=2, x=1 -> push 1     | heap nội bộ = [-1, -1, -1] | các viên = [1, 1, 1]
lượt 4: y=1, x=1 -> cả hai vỡ  | heap nội bộ = [-1] | các viên = [1]
Kết quả: 1


1

"heap nội bộ" là list số âm thật sự trong bộ nhớ — không sắp xếp hoàn toàn nhưng phần tử ở chỉ số 0 luôn là (âm của) viên nặng nhất.

### 6.1. Trace ca kết thúc bằng 0

In [12]:
trace_heap([2, 2, 3, 3])
print()
trace_heap([10, 4, 6])

bắt đầu: heap nội bộ = [-3, -3, -2, -2] | các viên = [3, 3, 2, 2]
lượt 1: y=3, x=3 -> cả hai vỡ  | heap nội bộ = [-2, -2] | các viên = [2, 2]
lượt 2: y=2, x=2 -> cả hai vỡ  | heap nội bộ = [] | các viên = []
Kết quả: 0

bắt đầu: heap nội bộ = [-10, -4, -6] | các viên = [10, 6, 4]
lượt 1: y=10, x=6 -> push 4     | heap nội bộ = [-4, -4] | các viên = [4, 4]
lượt 2: y=4, x=4 -> cả hai vỡ  | heap nội bộ = [] | các viên = []
Kết quả: 0


0

`[10,4,6]`: đập 10 và 6 → 4; còn `4, 4` → vỡ cả hai → `0`.

## 7. Code nộp LeetCode (Python 3)

In [13]:
class Solution:
    def lastStoneWeight(self, stones: list[int]) -> int:
        heap = [-s for s in stones]
        heapq.heapify(heap)
        while len(heap) > 1:
            y = -heapq.heappop(heap)
            x = -heapq.heappop(heap)
            if y != x:
                heapq.heappush(heap, x - y)      # = -(y - x)
        return -heap[0] if heap else 0

sol = Solution()
assert sol.lastStoneWeight([2, 7, 4, 1, 8, 1]) == 1
assert sol.lastStoneWeight([1]) == 1
print("Solution PASS 2 ví dụ của đề")

Solution PASS 2 ví dụ của đề


Bản sort mỗi lượt (đơn giản, đủ nhanh với `n ≤ 30`):

In [14]:
class SolutionSort:
    def lastStoneWeight(self, stones: list[int]) -> int:
        while len(stones) > 1:
            stones.sort()
            y, x = stones.pop(), stones.pop()
            if y != x:
                stones.append(y - x)
        return stones[0] if stones else 0

assert SolutionSort().lastStoneWeight([2, 7, 4, 1, 8, 1]) == 1
print("SolutionSort PASS")

SolutionSort PASS


`y, x = stones.pop(), stones.pop()`: vế phải được tính **từ trái sang phải** → `y` là pop thứ nhất (lớn nhất), `x` là pop thứ hai.

## 8. Vì sao đúng?

Đề mô tả một **quy trình xác định**: mỗi lượt đập đúng hai viên nặng nhất. Lời giải chỉ cần **mô phỏng chính xác** quy trình đó.

**Bất biến:** trước mỗi lượt, `heap` chứa (âm của) đúng các viên đá còn lại theo đề; `-heap[0]` là viên nặng nhất.
- `heappop` hai lần cho viên nặng nhất `y` và nặng nhì `x` (`x ≤ y`) — đúng cặp đề chọn (nếu có nhiều viên cùng cân nặng, chọn viên nào cũng như nhau vì chỉ cân nặng quan trọng).
- Đẩy `y - x` (nếu > 0) đúng như luật. Heap vẫn là heap sau `heappush` → bất biến được duy trì.

**Kết thúc:** mỗi lượt số viên giảm **ít nhất 1** (2 viên vào, tối đa 1 viên ra) → sau tối đa `n - 1` lượt, còn ≤ 1 viên. Vòng `while len(heap) > 1` dừng đúng lúc.

**Kết quả:** còn 1 viên → trả cân nặng; còn 0 viên → trả 0.

## 9. Phân tích Time / Space

Gọi `n = len(stones)`, `W = max(stones)`. Số lượt đập ≤ `n - 1`.

| Cách | Mỗi lượt | Tổng thời gian | Bộ nhớ thêm |
|---|---|---|---|
| 0. `max` + `remove` | O(n) | O(n²) | O(1) |
| 1. Sort mỗi lượt | O(n log n) | O(n² log n) | O(n) (Timsort) |
| 2. `insort` | O(n) | O(n²) | O(n) |
| 3. Max-heap | O(log n) | **O(n log n)** (heapify O(n)) | O(n) |
| 4. Bucket theo cân nặng | dò thùng | O(n·W) tệ nhất | O(W) |

Với constraints `n ≤ 30` mọi cách đều nhanh; heap là lời giải **mở rộng được** khi `n` lớn.

In [15]:
import random, time
random.seed(1046)
big = [random.randint(1, 10**9) for _ in range(20_000)]
for name, f in [("sort mỗi lượt", last_stone_sort), ("insort", last_stone_insort), ("heap", last_stone_heap)]:
    start = time.perf_counter()
    res = f(big)
    print(f"{name:<14}: kết quả {res:>10}, {time.perf_counter() - start:.3f}s")

sort mỗi lượt : kết quả          0, 0.718s
insort        : kết quả          0, 0.039s
heap          : kết quả          0, 0.011s


(Ngoài constraints để thấy khác biệt: `n = 20 000`, cân nặng tới `10^9`. Bucket không dùng được ở đây vì `W` quá lớn.)

## 10. Kiểm thử: mẫu đề, biên, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS ({actual})")

cases = [
    ("đề 1", [2, 7, 4, 1, 8, 1], 1),
    ("đề 2: một viên", [1], 1),
    ("hai viên bằng nhau", [3, 3], 0),
    ("hai viên khác nhau", [3, 10], 7),
    ("kết thúc bằng 0", [10, 4, 6], 0),
    ("toàn bằng nhau, số chẵn", [5, 5, 5, 5], 0),
    ("toàn bằng nhau, số lẻ", [5, 5, 5], 5),
    ("max constraints", [1000] * 29 + [1], 999),
]
solvers = {
    "sort": last_stone_sort,
    "insort": last_stone_insort,
    "heap": last_stone_heap,
    "heapreplace": last_stone_heapreplace,
    "bucket": last_stone_bucket,
    "Solution": lambda s: Solution().lastStoneWeight(s[:]),
    "SolutionSort": lambda s: SolutionSort().lastStoneWeight(s[:]),
}
for name, stones, exp in cases:
    for sname, f in solvers.items():
        assert f(stones) == exp, (name, sname)
    check(name, Solution().lastStoneWeight(stones[:]), exp)

đề 1                             PASS (1)
đề 2: một viên                   PASS (1)
hai viên bằng nhau               PASS (0)
hai viên khác nhau               PASS (7)
kết thúc bằng 0                  PASS (0)
toàn bằng nhau, số chẵn          PASS (0)
toàn bằng nhau, số lẻ            PASS (5)
max constraints                  PASS (999)


Ca "max constraints": 29 viên nặng 1000 → 28 viên tự triệt tiêu theo cặp, còn 1000 và 1 → 999.

`SolutionSort` **sửa trực tiếp** list đầu vào (giống LeetCode cho phép), nên khi test ta truyền bản sao `s[:]` để không làm hỏng `cases`.

### 10.1. Đối chiếu ngẫu nhiên (có seed) trong miền constraints

In [17]:
random.seed(2024)
for _ in range(3000):
    stones = [random.randint(1, random.choice([3, 20, 1000])) for _ in range(random.randint(1, 30))]
    expected = last_stone_sort(stones)
    for sname, f in solvers.items():
        assert f(stones) == expected, (sname, stones)
print("3000 bộ ngẫu nhiên: mọi cách khớp nhau")

3000 bộ ngẫu nhiên: mọi cách khớp nhau


`random.choice([3, 20, 1000])` chọn ngẫu nhiên **miền cân nặng** — miền nhỏ (`1..3`) tạo nhiều viên bằng nhau, giúp kiểm tra nhánh "cả hai vỡ".

### 10.2. Một tính chất: kết quả có **cùng tính chẵn lẻ** với tổng cân nặng

Mỗi lượt đập `x, y` thành `y - x` (hoặc 0): tổng giảm `2x` — số chẵn. Nên tổng ban đầu và kết quả cùng chẵn/lẻ.

In [18]:
for _ in range(2000):
    stones = [random.randint(1, 50) for _ in range(random.randint(1, 30))]
    assert Solution().lastStoneWeight(stones[:]) % 2 == sum(stones) % 2
print("Kết quả luôn cùng tính chẵn lẻ với tổng (2000 bộ)")

Kết quả luôn cùng tính chẵn lẻ với tổng (2000 bộ)


## 11. Sai lầm người mới hay mắc

### 11.1. Dùng `heapq` trực tiếp (min-heap) mà quên đổi dấu

In [19]:
def wrong_min_heap(stones):
    heap = stones[:]
    heapq.heapify(heap)
    while len(heap) > 1:
        a = heapq.heappop(heap)          # SAI: đây là hai viên NHẸ nhất
        b2 = heapq.heappop(heap)
        if a != b2:
            heapq.heappush(heap, abs(a - b2))
    return heap[0] if heap else 0

print("sai :", wrong_min_heap([2, 7, 4, 1, 8, 1]))
print("đúng:", Solution().lastStoneWeight([2, 7, 4, 1, 8, 1]))

sai : 3
đúng: 1


**Vì sao sai:** `heapq` lấy **nhỏ nhất**. Đề cần hai viên **nặng nhất**. **Sửa:** lưu `-s`.

### 11.2. Đổi dấu khi lưu nhưng quên đổi dấu khi push mảnh mới

In [20]:
def wrong_push_sign(stones):
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        y = -heapq.heappop(heap)
        x = -heapq.heappop(heap)
        if y != x:
            heapq.heappush(heap, y - x)      # SAI: đẩy số DƯƠNG vào heap số âm
    return -heap[0] if heap else 0

print("sai :", wrong_push_sign([2, 7, 4, 1, 8, 1]))
print("đúng:", Solution().lastStoneWeight([2, 7, 4, 1, 8, 1]))

sai : -1
đúng: 1


**Vì sao sai:** số dương `1` trong heap số âm bị coi là "viên nặng −1" — nhẹ nhất, và kết quả cuối bị đổi dấu sai. **Sửa:** `heappush(heap, -(y - x))`.

### 11.3. Trả `heap[0]` khi heap rỗng

In [21]:
def wrong_empty(stones):
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        y = -heapq.heappop(heap)
        x = -heapq.heappop(heap)
        if y != x:
            heapq.heappush(heap, -(y - x))
    return -heap[0]                          # SAI: heap có thể rỗng

try:
    wrong_empty([3, 3])
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().lastStoneWeight([3, 3]))

IndexError: list index out of range
đúng: 0


**Vì sao sai:** khi hai viên cuối bằng nhau, heap rỗng → `heap[0]` lỗi. **Sửa:** `return -heap[0] if heap else 0`.

### 11.4. Sắp xếp **một lần** rồi đập theo cặp từ cuối mà không cập nhật thứ tự

In [22]:
def wrong_sort_once(stones):
    arr = sorted(stones)
    while len(arr) > 1:
        y = arr.pop()
        x = arr.pop()
        if y != x:
            arr.append(y - x)                # SAI: mảnh mới có thể nhỏ, nhưng lại nằm ở cuối
    return arr[0] if arr else 0

print("sai :", wrong_sort_once([2, 7, 4, 1, 8, 1]))
print("đúng:", Solution().lastStoneWeight([2, 7, 4, 1, 8, 1]))

sai : -7
đúng: 1


**Vì sao sai:** sau khi `append(1)`, phần tử cuối là `1` — không còn là viên nặng nhất. Lượt sau lấy `y = 1`, `x = 4` (sai vai trò) và đẩy vào `1 - 4 = -3` — cân nặng **âm**, vô nghĩa; kết quả cuối thành `-7`. **Sửa:** sort lại mỗi lượt, `insort`, hoặc heap.

## 12. Nhận diện pattern

Dấu hiệu dùng **heap**:
- "lặp lại: lấy phần tử lớn nhất/nhỏ nhất, xử lý, đưa phần tử mới vào".
- "k phần tử lớn nhất/nhỏ nhất", "trộn k list đã sắp xếp", "lịch theo thời gian sớm nhất".
- Python: `heapq` là min-heap; cần max-heap thì **đổi dấu** (hoặc lưu tuple `(-key, ...)`).

Bài liên quan:
- LeetCode 2558 (Take Gifts From the Richest Pile) — max-heap + `k` lượt.
- LeetCode 215 (Kth Largest Element in an Array), 703 (Kth Largest Element in a Stream).
- LeetCode 1005 (Maximize Sum Of Array After K Negations) — min-heap lặp `k` lần.
- LeetCode 1049 (**Last Stone Weight II**) — tên giống nhưng **khác hẳn**: được chọn **bất kỳ** hai viên → bài toán chia tập (DP knapsack).

## 13. Biến thể

### 13.1. Biến thể A — LeetCode 2558: mỗi giây lấy đống quà nhiều nhất, để lại `floor(sqrt(x))`, sau `k` giây trả tổng còn lại

In [23]:
from math import isqrt

def pick_gifts(gifts: list[int], k: int) -> int:
    heap = [-g for g in gifts]
    heapq.heapify(heap)
    for _ in range(k):
        g = -heap[0]
        heapq.heapreplace(heap, -isqrt(g))
    return -sum(heap)

print(pick_gifts([25, 64, 9, 4, 100], 4))   # 29
print(pick_gifts([1, 1, 1, 1], 4))          # 4

29
4


`math.isqrt(n)` là căn bậc hai **nguyên** (làm tròn xuống), chính xác với số lớn — tốt hơn `int(n ** 0.5)`.

### 13.2. Biến thể B — trả về **nhật ký** các lượt đập (để debug/hiển thị)

In [24]:
def smash_log(stones: list[int]) -> list[tuple[int, int, int]]:
    heap = [-s for s in stones]
    heapq.heapify(heap)
    log = []
    while len(heap) > 1:
        y = -heapq.heappop(heap)
        x = -heapq.heappop(heap)
        log.append((y, x, y - x))
        if y != x:
            heapq.heappush(heap, -(y - x))
    return log

print(smash_log([2, 7, 4, 1, 8, 1]))

[(8, 7, 1), (4, 2, 2), (2, 1, 1), (1, 1, 0)]


## 14. Hệ thống luyện tập — 6 bài

### Bài 1 — Tính tay
Với `stones = [9, 3, 2, 10]`, lập bảng các lượt như mục 6 và cho đáp án.

### Bài 2 — Dự đoán output
Sau `h = [-5, -1, -8]; heapq.heapify(h)`, đoán `h[0]`, `-heapq.heappop(h)`, và `len(smash_log([4, 4, 4, 4]))`.

### Bài 3 — TODO: viết lại lời giải max-heap

In [25]:
def exercise_last_stone(stones: list[int]) -> int:
    # TODO 1: heap = [-s for s in stones]; heapify
    # TODO 2: while còn >= 2 viên: pop y, pop x; nếu khác thì push -(y - x)
    # TODO 3: trả -heap[0] nếu còn, ngược lại 0
    pass

# assert exercise_last_stone([2, 7, 4, 1, 8, 1]) == 1
# assert exercise_last_stone([3, 3]) == 0

### Bài 4 — TODO: số lượt đập cho tới khi kết thúc

In [26]:
def exercise_count_smashes(stones: list[int]) -> int:
    # TODO
    pass

# assert exercise_count_smashes([2, 7, 4, 1, 8, 1]) == 4
# assert exercise_count_smashes([1]) == 0
# assert exercise_count_smashes([5, 5, 5, 5]) == 2

### Bài 5 — TODO biến thể: đập hai viên **nhẹ nhất** thay vì nặng nhất (cùng luật còn lại)

In [27]:
def exercise_lightest_first(stones: list[int]) -> int:
    # TODO: min-heap, không cần đổi dấu
    pass

# assert exercise_lightest_first([2, 7, 4, 1, 8, 1]) == 3
# assert exercise_lightest_first([3, 3]) == 0

### Bài 6 — Mini challenge: Last Stone Weight II (LeetCode 1049) — được chọn **bất kỳ** hai viên; trả về cân nặng **nhỏ nhất có thể** của viên cuối

Gợi ý: kết quả cuối luôn có dạng `|S1 - S2|` với `S1 + S2 = tổng` (chia các viên thành hai nhóm "cộng" và "trừ"). Dùng `set` các tổng con có thể đạt được.

In [28]:
def exercise_last_stone_ii(stones: list[int]) -> int:
    # TODO
    pass

# assert exercise_last_stone_ii([2, 7, 4, 1, 8, 1]) == 1
# assert exercise_last_stone_ii([31, 26, 33, 21, 40]) == 5

## 15. Gợi ý ba mức

**Bài 1** — (1) bắt đầu `10 9 3 2`; (2) lượt 1 đập 10 và 9; (3) còn `3 2 1` → tiếp tục.

**Bài 2** — (1) heap lưu số âm; (2) `h[0]` là số âm nhỏ nhất; (3) 4 viên bằng nhau → 2 lượt.

**Bài 3** — (1) mục 5; (2) dấu khi push: `-(y - x)`; (3) `if heap else 0`.

**Bài 4** — (1) giống bài 3; (2) thêm biến `turns`; (3) `turns += 1` mỗi vòng `while`.

**Bài 5** — (1) `heapq` mặc định đã lấy nhỏ nhất; (2) pop `x` (nhẹ nhất), `y` (nhẹ nhì), `x <= y`; (3) push `y - x` nếu khác.

**Bài 6** — (1) `sums = {0}`, với mỗi viên `s`: `sums = {t + s for t in sums} | sums`; (2) muốn `S1` gần `tổng/2` nhất; (3) đáp án `min(abs(total - 2 * t) for t in sums)`.

## 16. Đáp án tham khảo

**Bài 1:** `10 9 3 2` → (10, 9) còn 1 → `3 2 1` → (3, 2) còn 1 → `1 1` → (1, 1) vỡ → **0**.

**Bài 2:** `h[0] = -8`, `-heappop(h) = 8`, `len(smash_log([4,4,4,4])) = 2`.

In [29]:
def answer_last_stone(stones):
    heap = [-s for s in stones]
    heapq.heapify(heap)
    while len(heap) > 1:
        y = -heapq.heappop(heap)
        x = -heapq.heappop(heap)
        if y != x:
            heapq.heappush(heap, -(y - x))
    return -heap[0] if heap else 0


def answer_count_smashes(stones):
    heap = [-s for s in stones]
    heapq.heapify(heap)
    turns = 0
    while len(heap) > 1:
        y = -heapq.heappop(heap)
        x = -heapq.heappop(heap)
        if y != x:
            heapq.heappush(heap, -(y - x))
        turns += 1
    return turns


def answer_lightest_first(stones):
    heap = stones[:]
    heapq.heapify(heap)
    while len(heap) > 1:
        x = heapq.heappop(heap)
        y = heapq.heappop(heap)
        if y != x:
            heapq.heappush(heap, y - x)
    return heap[0] if heap else 0


def answer_last_stone_ii(stones):
    total = sum(stones)
    sums = {0}
    for s in stones:
        sums = sums | {t + s for t in sums}
    return min(abs(total - 2 * t) for t in sums)


h = [-5, -1, -8]
heapq.heapify(h)
assert h[0] == -8 and -heapq.heappop(h) == 8 and len(smash_log([4, 4, 4, 4])) == 2
assert answer_last_stone([9, 3, 2, 10]) == 0
assert answer_last_stone([2, 7, 4, 1, 8, 1]) == 1 and answer_last_stone([3, 3]) == 0
assert answer_count_smashes([2, 7, 4, 1, 8, 1]) == 4
assert answer_count_smashes([1]) == 0 and answer_count_smashes([5, 5, 5, 5]) == 2
assert answer_lightest_first([2, 7, 4, 1, 8, 1]) == 3 and answer_lightest_first([3, 3]) == 0
assert answer_last_stone_ii([2, 7, 4, 1, 8, 1]) == 1
assert answer_last_stone_ii([31, 26, 33, 21, 40]) == 5

# kiểm tra Last Stone Weight II bằng vét cạn mọi cách gán dấu +/- (n nhỏ)
from itertools import product
random.seed(8)
for _ in range(300):
    stones = [random.randint(1, 20) for _ in range(random.randint(1, 8))]
    brute = min(abs(sum(sg * s for sg, s in zip(signs, stones))) for signs in product((1, -1), repeat=len(stones)))
    assert answer_last_stone_ii(stones) == brute
    assert answer_last_stone_ii(stones) <= answer_last_stone(stones)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


**Giải thích bài 6:** mỗi lần đập `y - x`, viên `x` mang dấu `−` và `y` mang dấu `+` vào kết quả. Cuối cùng, kết quả = tổng có dấu của mọi viên = `S1 − S2`. Ngược lại, mọi cách chia hai nhóm đều thực hiện được bằng một chuỗi đập hợp lệ. Vậy đáp án = `min |total − 2·S1|` trên các tổng con `S1` — bài toán **subset sum**. Kiểm tra cuối cùng cũng cho thấy: chơi theo luật "nặng nhất trước" (bài 1046) **không** đảm bảo kết quả nhỏ nhất.

## 17. Tổng kết — template + checklist

**Template nhớ nhanh (max-heap bằng số âm):**

```python
import heapq

class Solution:
    def lastStoneWeight(self, stones: list[int]) -> int:
        heap = [-s for s in stones]
        heapq.heapify(heap)
        while len(heap) > 1:
            y = -heapq.heappop(heap)
            x = -heapq.heappop(heap)
            if y != x:
                heapq.heappush(heap, -(y - x))
        return -heap[0] if heap else 0
```

**Câu thần chú:** *"Lặp lại lấy lớn nhất → heap. Python chỉ có min-heap → lưu số âm, nhớ đổi dấu cả lúc vào lẫn lúc ra."*

**Checklist trước khi submit:**
- [ ] Đổi dấu khi khởi tạo, khi pop, **và khi push** mảnh mới?
- [ ] Điều kiện vòng lặp `len(heap) > 1`?
- [ ] Heap rỗng → trả `0`?
- [ ] Hai viên bằng nhau → không push gì?
- [ ] Không nhầm với Last Stone Weight II (chọn bất kỳ — DP)?